# CIM Assignment 2 - Experiments with visual signals compression techniques


**Course**: Multimedia Information Codification  
**Academic Year**: 2025/2026    
**Authors**: Guilherme Rodrigues 202208878, João Oliveira 202205302         
**Date**: March 2026

## Importing libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os
import warnings
import pywt
from scipy.fft import dctn, idctn # To do Discrete Cosine Transform
from scipy.stats import entropy
from skimage.metrics import structural_similarity as ssim
from skimage import io, color
from skimage.metrics import peak_signal_noise_ratio as psnr


# Defining the paths and extensions
folder_path = './images_assignment2/'
extensions = ('.jpg', '.png', '.tif', '.bmp')

## TASK 1 - Experiments with techniques for assessing degree of spatial complexity, degree of similarity or redundancy in still images
### 1.1 - Basis 

Similarity to the first Visual Assignment, we create a function to divide the selected image to the various possible color spaces. This is the common foundations for the scripts throughout this Assignment


In [ ]:
# Create a list of images
image_files = [f for f in os.listdir(folder_path) if f.lower().endswith(extensions)] 

# Selection for different color spaces 
clr_space = 0 # 0 for YCrCb; 1 for LAB; 2 for VHS  

# Function responsible for the conversion
def convert(img_path,clr_space = 0):
    img_bgr = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    if (clr_space == 0):
        img_converted = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2YCrCb)
        names = 'YCrCb'
    elif (clr_space == 1):
        img_converted = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
        names = 'LAB'
    elif (clr_space == 2):
        img_converted = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
        names = 'HSV'

    return cv2.split(img_converted), names, img_rgb



### 1.1.i. Applying wavelet transforms

- This code cell contains the functions responsible to do a transformation to an image dividing (in this 1st order case) in 4 regions, where is determined if a Low or High pass filter was applied to it. 
- Applying a Lowpass filter both horizontally and Vertically will essentially preserve the original's image (the low frequency components will remain visible)
- Applying a Highpass filter in a certain direction will enlighten the said direction edges (the best example is in the main code down below, where an Highpass filter is applied both horizontally and vertically, enhancing the diagonal edges).
- HL will make vertical edges noticeable, and LH will make horizontal edges noticeable.


In [ ]:
def analyze_wavelet(lum_channel, wavelet_name='haar'):
  coeffs = pywt.dwt2(lum_channel, wavelet_name) # This transformation will lead to
  # -------------
  # | LL  | HL  |   
  #  -----------                                          
  # | LH  | HH  |
  # -------------
  LL, (LH, HL, HH) = coeffs   

  energy = {
    'LL': np.sum(LL**2),
    'LH': np.sum(LH**2),
    'HL': np.sum(HL**2),
    'HH': np.sum(HH**2)
  }     

  total_energy = sum(energy.values())   

  normalized_energy = {k: v / total_energy for k, v in energy.items()} # "Normalisation can be done by dividing the energy of each channel by the total energy across all channels"

  return normalized_energy,coeffs
                                                 

### 1.1.ii) Structural similarity index of an image

This function serves to apply SSIM, to evaluate the similarity between, taking the lumminance channel as an input and doing a spatial correlation on it. 
- We divide the image in 32 regions, and then we will iterate through each region;
- From each region the top left corner (an 8x8 block) is considered the reference;
- Inside said region, we will pick the other 3 8x8 blocks and calculate its SSIM (function similar to the one seen in the UC's slides). This value is stored for every iteration;
- Upon this correlation for every region, all SSIM scores are outputted, exporting a map containing the results. A low SSIM value corresponds to a high complexity region;
- The plot pixels are white for a high SSIM (present in components like the sky or the sead, since they not represent a high complexity zone), and black when SSIM is low.

In [ ]:
def analyze_ssim (lum_channel):
    height, width = lum_channel.shape

    height_region, width_region = height//32, width//32 # Dividing in blocks of 32x32
    ssim_map = np.zeros((height_region,width_region))

    for i in range(height_region):
        for j in range(width_region):
            region = lum_channel[i*32:(i+1)*32, j*32:(j+1)*32] # Extracting the 32x32 values, in other sections it will be done the same but with 8x8

            ref_block = region[0:8,0:8] # Reference -> Top-Left Corner

            local_scores = []

            for y in range (0,25,8): # y = [0, 7, 15, 23]
                for x in range (0,25,8): # x = [0, 7, 15, 23]
                    target_block = region[y:y+8, x:x+8] # 0:7;8:15;16:23;24:31 -> The regions covered by each iteration y wise and x wise

                    score = ssim(ref_block, target_block, data_range=255)
                    local_scores.append(score)
            
            ssim_map[i,j] = np.mean(local_scores) # The similarity of this iteration's region
    
    return ssim_map # Low SSIM = High Complexity


### 1.1.iii) Histogram Analysis

This function is used to analyze the image though an histogram, where the pixel intensities are divided from 0 to 255 (corresponding to 256 bins).

- The luminance channel is decribed following a histogram;
- It's then normalized so the image values, when added together, equal 1;
- The complexity is then calculated via Shannon Entropy's formula;
- When high entropy values are obtained, the more complex the image is.

In [ ]:
def analyze_histogram (lum_channel):
    hist, _ = np.histogram(lum_channel, bins=256, range=(0,255))

    prob_dist = hist / hist.sum()

    complexity_score = entropy(prob_dist,base=2) # Shannon

    return complexity_score # Higher entropy = More Complex

### 1.1) Main code

In [ ]:
technique = 0 # 0 for Wavelet; 1 for SSIM; 2 for Histogram

for filename in image_files:
    path = os.path.join(folder_path, filename)

    channels, titles, original_img = convert(path, clr_space)

    Y = channels[0]

    for technique in range(3):  # 0, 1, 2
        plt.figure(figsize=(15, 5))
        
        # Show original image
        plt.subplot(1, 2, 1)
        plt.imshow(original_img)
        plt.title(f"Original Image: {filename}")
        plt.axis('off')
        
        # Show technique result
        plt.subplot(1, 2, 2)

        # 1.1.i)
        if technique == 0:
            energy, coeffs = analyze_wavelet(Y)
            LL, (LH, HL, HH) = coeffs
            print(f"Detail Energy (HH) (normalised): {energy['HH']:.4f}")
        
            plt.imshow(HH, cmap='jet')
            plt.title(f"Wavelet HH Energy (normalised): {energy['HH']:.4f}")

        # 1.1.ii)
        elif technique == 1:
            s_map = analyze_ssim(Y)
            avg_sim = np.mean(s_map)
            print(f"Average Image Similarity: {avg_sim:.4f}")
            
            plt.imshow(s_map, cmap='magma')
            plt.title(f"SSIM Map (Avg: {avg_sim:.2f})")

        # 1.1.iii)
        elif technique == 2:
            ent_score = analyze_histogram(Y)
            print(f"Image Entropy: {ent_score:.4f}")
            
            plt.hist(Y.ravel(), bins=256, color='gray')
            plt.title(f"Histogram (Entropy: {ent_score:.2f})")

        plt.suptitle(f"Complexity Analysis: {filename}")
        plt.tight_layout()
        #plt.colorbar()
        plt.show()

## TASK 2 - Experiments with transform coding and quantisation
### 2.1 - Basis 
- We will use the initial convert() function created in 1.1, since it serves the exact same purpose.

### 2.1.i. Creating DCT transformated blocks

This function divides the luminance channel into a grid of 8×8 pixel blocks and transforms them from the spatial domain to the frequency domain.

- Block Partitioning: We iterate through the image in steps of 8 pixels, isolating each individual block.

- Discrete Cosine Transform (DCT): We apply a 2D DCT (orthogonal variant) to each block. This is mathematically equivalent to the operation T⋅B⋅TT, where B is the image block and T is the DCT basis matrix.

- Frequency Mapping: The resulting coefficients represent the "spectral" content of the block.

    - The DC coefficient (top-left) represents the average brightness.

    - The AC coefficients represent patterns of change: horizontal frequencies increase as you move right, and vertical frequencies increase as you move down.

In [ ]:
def get_dct_blocks(lum_channel):
    height, width = lum_channel.shape # The image should be a multiple of 8
    h_blocks, w_blocks = height//8, width//8
    dct_blocks = np.zeros((h_blocks,w_blocks,8,8))

    for i in range(h_blocks):
        for j in range(w_blocks):
            block = lum_channel[i*8:(i+1)*8, j*8:(j+1)*8].astype(float)
            dct_blocks[i, j] = dctn(block, norm='ortho') # The same as doing T.Block.T'. Orthogonalized variant
            
    return dct_blocks, (h_blocks, w_blocks)


### 2.1.ii. Computing the energy contained in the DCT coefficients

This functions, creates a ZigZag pathern to "travel" through the block, and at the same time describes the energy concentration of the coefficients.

- The ZigZag sequence will make sure we start analyzing the most important parts (from the low frequency area to the highest frequency area). 
- Upon transforming our 2D block into a 1D block, following the ZigZag path, we then square the coefficients, and sum all their values. 
- Now, having the total sum obtained, we obtain the cummulative sum array which allow us to detect the first coefficient energy  value that surpasses the threshold value.
- This index serves as the output of the function. 


In [ ]:

def zigzag_order():
    return [
    (0,0), (0,1), (1,0), (2,0), (1,1), (0,2), (0,3), (1,2), (2,1), (3,0), (4,0), (3,1), (2,2), (1,3), (0,4), (0,5),
    (1,4), (2,3), (3,2), (4,1), (5,0), (6,0), (5,1), (4,2), (3,3), (2,4), (1,5), (0,6), (0,7), (1,6), (2,5), (3,4),
    (4,3), (5,2), (6,1), (7,0), (7,1), (6,2), (5,3), (4,4), (3,5), (2,6), (1,7), (2,7), (3,6), (4,5), (5,4), (6,3),
    (7,2), (7,3), (6,4), (5,5), (4,6), (3,7), (4,7), (5,6), (6,5), (7,4), (7,5), (6,6), (5,7), (6,7), (7,6), (7,7)
    ]



def first_k(dct_blocks, E_threshold):
    height_b, width_b, _, _ = dct_blocks.shape
    k_values = []

    ZIGZAG_ORDER = zigzag_order()

    for i in range(height_b):
        for j in range(width_b):
            block = dct_blocks[i,j]
            coeffs = np.array([block[pos] for pos in ZIGZAG_ORDER]) # Spreading the coefficients in zig-zag order
            energies = coeffs**2
            total_block_energy = np.sum(energies) # Energy= Summation ||Cij||2 ?

            cumulative_energy = np.cumsum(energies)

            k = np.where(cumulative_energy >= E_threshold*total_block_energy)[0][0] + 1 # [0][0] to the minimum k
            k_values.append(k)

    return int(np.mean(k_values))

### 2.1.iii. Inverse transform and Low-Pass filtering



This function reconstructs the luminance channel using only the most significant k coefficients for each block:

- Frequency Truncation (k-selection): For every 8×8 block, we take the coefficients in zigzag order and keep only the first k values. We set all other coefficients (from k to 64) to zero.

- Low-Pass Filtering: Because the zigzag order starts with low frequencies (broad shapes) and ends with high frequencies (fine details), keeping only the first k coefficients acts as a low-pass filter.

- Inverse DCT (IDCT): We map the truncated coefficients back into a 2D grid and apply the Inverse Discrete Cosine Transform. This converts the data from the frequency domain back into the spatial (pixel) domain.

- Image Reassembly: The blocks are stitched back together to form the recovered image.

In [ ]:
def reconstruct_image (dct_blocks, k, h_blocks, w_blocks):
    recovered = np.zeros((h_blocks*8, w_blocks*8))

    ZIGZAG_ORDER = zigzag_order()

    for i in range(h_blocks):
        for j in range(w_blocks):
            block = dct_blocks[i, j]
            coeffs = np.array([block[pos] for pos in ZIGZAG_ORDER])
            compressed_coeffs = np.zeros(64)
            compressed_coeffs[:k] = coeffs[:k] # To select the first k elements

            # Put back into 8x8 block 
            new_block = np.zeros((8,8))
            for idx, pos in enumerate(ZIGZAG_ORDER):
                new_block[pos] = compressed_coeffs[idx]
            
            # Inverse DCT
            recovered[i*8:(i+1)*8, j*8:(j+1)*8] = idctn(new_block, norm='ortho') 

    return recovered

### 2.1. Main code


In [ ]:
E_threshold = 0.95

for filename in image_files:
    # 2.1 Basis
    path = os.path.join(folder_path, filename)
    channels, titles, original_img = convert(path, clr_space)
    Y = channels[0]

    # 2.1.i)
    dct_blocks, (h_b, w_b) = get_dct_blocks(Y)

    # 2.1.ii) 
    k_to_use = first_k(dct_blocks, E_threshold)
    print(f"For {filename}, keeping {k_to_use}/64 coefficients to reach {E_threshold*100}% energy.")

    #2.1.iii)
    recovered = reconstruct_image(dct_blocks, k_to_use, h_b, w_b)

    # Merge back for display 
    h_final, w_final = recovered.shape

    final_img_pre_rgb = cv2.merge([
        recovered.clip(0, 255).astype(np.uint8), # Just to make sure the values do not surpass 2^8
        channels[1][:h_final, :w_final],
        channels[2][:h_final, :w_final]
    ])

    # Just to make sure the format is converted in the end correctly
    if clr_space == 0 :
        final_img = cv2.cvtColor(final_img_pre_rgb, cv2.COLOR_YCrCb2RGB)
    elif clr_space == 1:
        final_img = cv2.cvtColor(final_img_pre_rgb, cv2.COLOR_LAB2RGB)
    elif clr_space == 2:
        final_img = cv2.cvtColor(final_img_pre_rgb, cv2.COLOR_HSV2RGB)

    # Plotting
    plt.figure(figsize=(10, 5))
    plt.title(titles)
    plt.subplot(1, 2, 1); plt.imshow(original_img); plt.title("Original "); plt.axis("off")
    plt.subplot(1, 2, 2); plt.imshow(final_img); plt.title(f"Recovered (k={k_to_use})"); plt.axis("off")
    plt.show()

### 2.2 

In [ ]:
def jpeg_quantization_matrix(quality=50):
    """Generate JPEG quantization matrix scaled by quality factor"""
    # Default JPEG luminance quantization table
    base_matrix = np.array([
        [16, 11, 10, 16, 24, 40, 51, 61],
        [12, 12, 14, 19, 26, 58, 60, 55],
        [14, 13, 16, 24, 40, 57, 69, 56],
        [14, 17, 22, 29, 51, 87, 80, 62],
        [18, 22, 37, 56, 68, 109, 103, 77],
        [24, 35, 55, 64, 81, 104, 113, 92],
        [49, 64, 78, 87, 103, 121, 120, 101],
        [72, 92, 95, 98, 112, 100, 103, 99]
    ])
    
    # Scale based on quality factor (JPEG standard)
    if quality < 50:
        scale = 5000 / quality
    else:
        scale = 200 - 2 * quality
    
    scaled_matrix = np.floor((base_matrix * scale + 50) / 100)
    scaled_matrix[scaled_matrix == 0] = 1  # Avoid division by zero
    
    return scaled_matrix.astype(int)

def select_quality_from_complexity(complexity_val, method_used):
    """Select quality factor based on spatial complexity"""
    # Map complexity to quality factor
    # Higher complexity -> higher quality (less compression)
    if method_used == 0:  # Wavelet - HH energy
        # HH energy around 0.01-0.3 typically
        quality = 30 + (complexity_val * 200)  # Scale to 30-90 range
    elif method_used == 1:  # SSIM - lower SSIM = higher complexity
        # SSIM map average around 0.3-0.9
        complexity = 1 - complexity_val  # Invert so higher = more complex
        quality = 30 + (complexity * 70)
    elif method_used == 2:  # Histogram entropy
        # Entropy around 4-7 for typical images
        complexity_norm = complexity_val / 8.0  # Normalize by max entropy (8)
        quality = 30 + (complexity_norm * 70)
    
    return int(np.clip(quality, 10, 95))


def quantize_block(dct_block, q_matrix):
    """Quantize DCT coefficients using JPEG matrix"""
    return np.round(dct_block / q_matrix)

def inverse_quantize_block(quant_block, q_matrix):
    """Inverse quantization"""
    return quant_block * q_matrix

def apply_threshold(quant_block, threshold):
    """Zero out coefficients below threshold"""
    quant_block_copy = quant_block.copy()
    quant_block_copy[np.abs(quant_block_copy) < threshold] = 0
    return quant_block_copy

def count_nonzero_coeffs(quant_block):
    """Count non-zero coefficients in quantized block"""
    return np.count_nonzero(quant_block)



In [ ]:
def jpeg_like_compression_adaptive(filename, folder_path, clr_space=0, 
                                  complexity_method=0, fixed_quality=None, 
                                  threshold=5, E_threshold_show=False):
    """
    Main function for 2.2 implementing adaptive JPEG-like compression
    """
    
    print(f"\n{'='*60}")
    print(f"Processing: {filename}")
    print(f"{'='*60}")
    
    # 1. Import and convert image (using your existing convert function)
    path = os.path.join(folder_path, filename)
    channels, titles, original_img = convert(path, clr_space)
    Y = channels[0]  # Luminance component
    
    # 2. Assess spatial complexity (using your existing functions)
    print("\n--- Spatial Complexity Assessment ---")
    if complexity_method == 0:  # Wavelet
        energy_norm, coeffs = analyze_wavelet(Y)
        complexity_val = energy_norm['HH']  # Use HH band energy
        method_name = "Wavelet (HH Energy)"
        print(f"  Method: {method_name}")
        print(f"  Complexity value (HH energy): {complexity_val:.4f}")
        
    elif complexity_method == 1:  # SSIM
        s_map = analyze_ssim(Y)
        complexity_val = np.mean(s_map)  # Average similarity
        method_name = "SSIM (Avg Similarity)"
        print(f"  Method: {method_name}")
        print(f"  Complexity value (Avg SSIM): {complexity_val:.4f}")
        print(f"  Note: Lower SSIM = Higher complexity")
        
    elif complexity_method == 2:  # Histogram
        complexity_val = analyze_histogram(Y)
        method_name = "Histogram Entropy"
        print(f"  Method: {method_name}")
        print(f"  Complexity value (Entropy): {complexity_val:.4f}")
    
    # 3. Select quality factor (adaptive or fixed)
    if fixed_quality is None:
        quality_factor = select_quality_from_complexity(complexity_val, complexity_method)
        print(f"\n--- Adaptive Quality Selection ---")
        print(f"  Selected quality factor: {quality_factor}")
    else:
        quality_factor = fixed_quality
        print(f"\n--- Fixed Quality ---")
        print(f"  Using fixed quality factor: {quality_factor}")
    
    # 4. Get DCT blocks (using your existing function)
    print(f"\n--- Processing 8x8 Blocks ---")
    dct_blocks, (h_blocks, w_blocks) = get_dct_blocks(Y)
    
    # 5. Get JPEG quantization matrix
    q_matrix = jpeg_quantization_matrix(quality_factor)
    
    # 6. Process each block: Quantize -> Threshold -> Inverse Quantize
    quantized_blocks = np.zeros_like(dct_blocks)
    dequantized_blocks = np.zeros_like(dct_blocks)
    
    total_coeffs = 0
    total_nonzero = 0
    
    for i in range(h_blocks):
        for j in range(w_blocks):
            # Get block
            dct_block = dct_blocks[i, j]
            
            # Quantize
            quant_block = quantize_block(dct_block, q_matrix)
            
            # Apply threshold
            quant_block = apply_threshold(quant_block, threshold)
            
            # Count non-zero coefficients
            nz = count_nonzero_coeffs(quant_block)
            total_nonzero += nz
            total_coeffs += 64
            
            # Store quantized block
            quantized_blocks[i, j] = quant_block
            
            # Inverse quantize
            dequantized_blocks[i, j] = inverse_quantize_block(quant_block, q_matrix)
    
    compression_ratio = total_coeffs / max(total_nonzero, 1)
    print(f"  Total coefficients: {total_coeffs}")
    print(f"  Non-zero coefficients: {total_nonzero}")
    print(f"  Compression ratio (based on coeffs): {compression_ratio:.2f}:1")
    
    # 7. Reconstruct image from dequantized blocks
    # Reshape dequantized_blocks back to image
    recovered_Y = np.zeros((h_blocks*8, w_blocks*8))
    
    for i in range(h_blocks):
        for j in range(w_blocks):
            block = dequantized_blocks[i, j]
            # Inverse DCT
            recovered_Y[i*8:(i+1)*8, j*8:(j+1)*8] = idctn(block, norm='ortho')
    
    # Clip to valid range
    recovered_Y = np.clip(recovered_Y, 0, 255)
    
    # 8. Merge back with color channels
    h_final, w_final = recovered_Y.shape
    
    final_img_pre_rgb = cv2.merge([
        recovered_Y.astype(np.uint8),
        channels[1][:h_final, :w_final],
        channels[2][:h_final, :w_final]
    ])
    
    # Convert back to RGB based on color space
    if clr_space == 0:
        final_img = cv2.cvtColor(final_img_pre_rgb, cv2.COLOR_YCrCb2RGB)
    elif clr_space == 1:
        final_img = cv2.cvtColor(final_img_pre_rgb, cv2.COLOR_LAB2RGB)
    elif clr_space == 2:
        final_img = cv2.cvtColor(final_img_pre_rgb, cv2.COLOR_HSV2RGB)

    if Y.shape != recovered_Y.shape:
        print("Shapes don't match! Adjusting...")
        # Crop to the smallest dimensions
        min_h = min(Y.shape[0], recovered_Y.shape[0])
        min_w = min(Y.shape[1], recovered_Y.shape[1])
        Y = Y[:min_h, :min_w]
        recovered_Y = recovered_Y[:min_h, :min_w]
    
    # 9. Quality evaluation
    # Convert to float for metrics
    Y_float = Y.astype(float)
    recovered_float = recovered_Y.astype(float)
    
    # Normalize for metrics
    Y_norm = (Y_float - Y_float.min()) / (Y_float.max() - Y_float.min() + 1e-10)
    recovered_norm = (recovered_float - recovered_float.min()) / (recovered_float.max() - recovered_float.min() + 1e-10)
    
    psnr_val = psnr(Y_norm, recovered_norm, data_range=1)
    ssim_val = ssim(Y_norm, recovered_norm, data_range=1)
    
    print(f"\n--- Quality Assessment ---")
    print(f"  PSNR: {psnr_val:.2f} dB")
    print(f"  SSIM: {ssim_val:.4f}")
    
    # 10. Display results
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # Original image
    axes[0, 0].imshow(original_img)
    axes[0, 0].set_title(f"Original Image\n{filename}")
    axes[0, 0].axis('off')
    
    # Original luminance
    axes[0, 1].imshow(Y, cmap='gray')
    axes[0, 1].set_title("Luminance (Y) Channel")
    axes[0, 1].axis('off')
    
    # Recovered image
    axes[0, 2].imshow(final_img)
    axes[0, 2].set_title(f"Recovered Image\n(Q={quality_factor}, Th={threshold})")
    axes[0, 2].axis('off')
    
    # Recovered luminance
    axes[1, 0].imshow(recovered_Y, cmap='gray')
    axes[1, 0].set_title(f"Recovered Luminance\nPSNR: {psnr_val:.2f}dB")
    axes[1, 0].axis('off')
    
    # Quantization matrix
    axes[1, 1].imshow(q_matrix, cmap='viridis')
    axes[1, 1].set_title(f"Quantization Matrix\n(Quality={quality_factor})")
    axes[1, 1].axis('off')
    
    # Error map
    error_map = np.abs(Y.astype(float) - recovered_Y.astype(float))
    im = axes[1, 2].imshow(error_map, cmap='hot')
    axes[1, 2].set_title(f"Error Map\nSSIM: {ssim_val:.4f}")
    axes[1, 2].axis('off')
    plt.colorbar(im, ax=axes[1, 2])
    
    # Add complexity info as text
    info_text = f"Spatial Complexity Assessment:\n"
    info_text += f"Method: {method_name}\n"
    info_text += f"Complexity value: {complexity_val:.4f}\n"
    info_text += f"Quality factor: {quality_factor} ({'Adaptive' if fixed_quality is None else 'Fixed'})\n"
    info_text += f"Threshold: {threshold}\n"
    info_text += f"Compression ratio: {compression_ratio:.2f}:1"
    
    plt.suptitle(info_text, fontsize=10)
    plt.tight_layout()
    plt.show()
    
    return {
        'filename': filename,
        'complexity_method': complexity_method,
        'complexity_val': complexity_val,
        'quality_factor': quality_factor,
        'threshold': threshold,
        'psnr': psnr_val,
        'ssim': ssim_val,
        'compression_ratio': compression_ratio,
        'nonzero_coeffs': total_nonzero,
        'total_coeffs': total_coeffs
    }

In [ ]:
def run_experiments_22(image_files, folder_path, clr_space=0):
    """
    Run experiments with different settings as required in 2.2
    """
    
    all_results = []
    
    for filename in image_files:
        print(f"\n\n{'#'*70}")
        print(f"#" + f" EXPERIMENTS FOR: {filename} ".center(68) + "#")
        print(f"{'#'*70}")
        
        # Test with different complexity methods (adaptive)
        for method in range(3):
            method_names = ["Wavelet", "SSIM", "Histogram"]
            print(f"\n--- Adaptive with {method_names[method]} complexity ---")
            result = jpeg_like_compression_adaptive(
                filename, folder_path, clr_space=clr_space,
                complexity_method=method, fixed_quality=None, threshold=5
            )
            result['test_type'] = f"Adaptive-{method_names[method]}"
            all_results.append(result)
        
        # Test with fixed quality (no adaptation)
        for quality in [30, 50, 70]:
            print(f"\n--- Fixed Quality = {quality} ---")
            result = jpeg_like_compression_adaptive(
                filename, folder_path, clr_space=clr_space,
                complexity_method=0, fixed_quality=quality, threshold=5
            )
            result['test_type'] = f"Fixed-Q{quality}"
            all_results.append(result)
        
        # Test with different thresholds
        for thresh in [0, 2, 10, 20]:
            print(f"\n--- Threshold = {thresh} (Adaptive Wavelet) ---")
            result = jpeg_like_compression_adaptive(
                filename, folder_path, clr_space=clr_space,
                complexity_method=0, fixed_quality=None, threshold=thresh
            )
            result['test_type'] = f"Thresh-{thresh}"
            all_results.append(result)
    
    # Print summary table
    print("\n\n" + "="*100)
    print("SUMMARY OF RESULTS".center(100))
    print("="*100)
    print(f"{'Filename':<15} {'Test Type':<20} {'Complexity':<10} {'Q':<5} {'Th':<5} {'PSNR':<8} {'SSIM':<8} {'Comp Ratio':<10}")
    print("-"*100)
    
    for r in all_results:
        print(f"{r['filename']:<15} {r['test_type']:<20} {r['complexity_val']:<10.3f} "
              f"{r['quality_factor']:<5} {r['threshold']:<5} {r['psnr']:<8.2f} "
              f"{r['ssim']:<8.4f} {r['compression_ratio']:<10.2f}")
    
    return all_results

results = run_experiments_22(image_files, folder_path, clr_space)